# 🤖 Hackathon #26 — Modélisation IA & Projections Climatiques
## Notebook 02 : Entraînement, Benchmark et Scénarios

**Objectif** : Entraîner, comparer et interpréter les modèles prédictifs climatiques.

**Plan** :
1. Préparation des séries temporelles
2. Entraînement des 4 modèles (ARIMA, Prophet, LSTM, XGBoost)
3. Benchmark comparatif (RMSE, MAE, MAPE, R²)
4. Projections 2030 / 2050 / 2100 selon 3 scénarios GIEC
5. Interprétabilité (feature importance XGBoost)
6. Visualisation des intervalles de confiance


In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from config.settings import PROCESSED_DIR, REPORTS_DIR, SCENARIOS, PROJECTION_YEARS
from src.models.all_models import ARIMAModel, ProphetModel, LSTMModel, GradientBoostingModel
from src.models.model_comparison import ModelBenchmark

print('✅ Imports OK')

✅ Imports OK


## 1. Chargement et préparation des données

In [2]:
# Chargement master dataset
from src.processing.transformer import DataTransformer

master_path = PROCESSED_DIR / 'master_features.parquet'
if not master_path.exists():
    print('⚙️ Génération du dataset...')
    transformer = DataTransformer()
    transformer.run()

df = pd.read_parquet(master_path)
print(f'Dataset : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print(f'Période : {df["annee"].min()} → {df["annee"].max()}')
df[['annee', 'temp_moy_c', 'co2_ppm', 'anomalie_temp_c', 'score_risque_climatique']].tail(10)

Dataset : 125 lignes × 28 colonnes
Période : 1900 → 2024


,annee,temp_moy_c,co2_ppm,anomalie_temp_c,score_risque_climatique
115,2015,13.6500,401.01,0.581,66.6
116,2016,13.5800,404.41,0.511,68.6
117,2017,13.3100,406.76,0.241,64.3
118,2018,13.9000,408.72,0.831,74.1
119,2019,15.1696,411.65,2.101,98.9
120,2020,13.8400,414.21,0.771,70.6
121,2021,13.4300,416.41,0.361,67.8
122,2022,15.1696,418.53,2.101,99.5
123,2023,13.3400,421.08,0.271,65.8
124,2024,13.8500,424.61,0.781,72.2


In [3]:
# Visualisation train/test split
TARGET = 'temp_moy_c'
TEST_SIZE = 0.2
split_idx = int(len(df) * (1 - TEST_SIZE))
split_year = df['annee'].iloc[split_idx]

df_clean = df.dropna(subset=[TARGET])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_clean['annee'][:split_idx], y=df_clean[TARGET][:split_idx],
    mode='lines', name='Train', line=dict(color='#6c5ce7', width=2),
))
fig.add_trace(go.Scatter(
    x=df_clean['annee'][split_idx:], y=df_clean[TARGET][split_idx:],
    mode='lines', name='Test', line=dict(color='#fd79a8', width=2),
))
fig.add_vline(x=split_year, line_dash='dash', line_color='yellow',
              annotation_text=f'Split : {split_year:.0f}')
fig.update_layout(
    title=f'📊 Train / Test Split — {TARGET}',
    template='plotly_dark', height=350,
    xaxis_title='Année', yaxis_title='°C',
)
fig.show()

## 2. Entraînement des modèles

In [4]:
# ── ARIMA / SARIMA ──────────────────
print('📈 Entraînement SARIMA...')
arima = ARIMAModel(target=TARGET)
metrics_arima = arima.train(df, test_size=TEST_SIZE)
print(f'  RMSE: {metrics_arima["rmse"]:.4f} | MAE: {metrics_arima["mae"]:.4f} | R²: {metrics_arima["r2"]:.4f}')

2026/03/16 10:39:22 INFO mlflow.tracking.fluent: Experiment with name 'hackathon26_temp_moy_c' does not exist. Creating a new experiment.
2026-03-16 10:39:22.378 | INFO     | src.models.base_model:train:63 - 🤖 [SARIMA] Entraînement sur 'temp_moy_c'...


📈 Entraînement SARIMA...


2026-03-16 10:39:44.160 | INFO     | src.models.all_models:_fit:59 -   SARIMA ordre : (5, 1, 1) × (0, 0, 0, 12)
2026-03-16 10:39:44.169 | SUCCESS  | src.models.base_model:train:86 - ✅ [SARIMA] RMSE=0.5850 | MAE=0.3447 | R²=-0.0298


  RMSE: 0.5850 | MAE: 0.3447 | R²: -0.0298


In [5]:
# ── Prophet ─────────────────────────
print('🔮 Entraînement Prophet...')
prophet = ProphetModel(target=TARGET)
metrics_prophet = prophet.train(df, test_size=TEST_SIZE)
print(f'  RMSE: {metrics_prophet["rmse"]:.4f} | MAE: {metrics_prophet["mae"]:.4f} | R²: {metrics_prophet["r2"]:.4f}')

2026-03-16 10:39:44.180 | INFO     | src.models.base_model:train:63 - 🤖 [Prophet] Entraînement sur 'temp_moy_c'...
10:39:44 - cmdstanpy - INFO - Chain [1] start processing


🔮 Entraînement Prophet...


10:39:45 - cmdstanpy - INFO - Chain [1] done processing
2026-03-16 10:39:45.577 | INFO     | src.models.all_models:_fit:131 -   Prophet entraîné sur 100 années
2026-03-16 10:39:45.604 | SUCCESS  | src.models.base_model:train:86 - ✅ [Prophet] RMSE=0.5930 | MAE=0.3484 | R²=-0.0582


  RMSE: 0.5930 | MAE: 0.3484 | R²: -0.0582


In [ ]:
# ── LSTM ────────────────────────────
print('🧠 Entraînement LSTM...')
lstm = LSTMModel(target=TARGET)
metrics_lstm = lstm.train(df, test_size=TEST_SIZE)
print(f'  RMSE: {metrics_lstm["rmse"]:.4f} | MAE: {metrics_lstm["mae"]:.4f} | R²: {metrics_lstm["r2"]:.4f}')

2026-03-16 10:39:45.612 | INFO     | src.models.base_model:train:63 - 🤖 [LSTM] Entraînement sur 'temp_moy_c'...


In [ ]:
# ── XGBoost ─────────────────────────
print('🌲 Entraînement XGBoost...')
xgb_model = GradientBoostingModel(target=TARGET)
metrics_xgb = xgb_model.train(df, test_size=TEST_SIZE)
print(f'  RMSE: {metrics_xgb["rmse"]:.4f} | MAE: {metrics_xgb["mae"]:.4f} | R²: {metrics_xgb["r2"]:.4f}')

## 3. Benchmark comparatif

In [ ]:
# Tableau comparatif
all_metrics = {
    'SARIMA': metrics_arima,
    'Prophet': metrics_prophet,
    'LSTM': metrics_lstm,
    'XGBoost': metrics_xgb,
}

df_metrics = pd.DataFrame(all_metrics).T
df_metrics['Rang_RMSE'] = df_metrics['rmse'].rank().astype(int)

# Mise en forme
best_model = df_metrics['rmse'].idxmin()
print(f'🏆 Meilleur modèle (RMSE) : {best_model}')
df_metrics.sort_values('rmse')

In [ ]:
# Visualisation radar des métriques
categories = ['RMSE (inv)', 'MAE (inv)', 'R²']
models_plot = list(all_metrics.keys())
colors_radar = ['#6c5ce7', '#fd79a8', '#fdcb6e', '#00cec9']

fig_radar = go.Figure()
for i, (model_name, m) in enumerate(all_metrics.items()):
    # Normalisation : RMSE et MAE inversés (plus bas = meilleur → plus haut sur radar)
    rmse_inv = 1 - (m['rmse'] / max(v['rmse'] for v in all_metrics.values()))
    mae_inv = 1 - (m['mae'] / max(v['mae'] for v in all_metrics.values()))
    r2_norm = max(0, m['r2'])
    
    values = [rmse_inv, mae_inv, r2_norm]
    fig_radar.add_trace(go.Scatterpolar(
        r=values + [values[0]],
        theta=categories + [categories[0]],
        fill='toself',
        name=model_name,
        line_color=colors_radar[i],
        opacity=0.7,
    ))

fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='🎯 Radar des performances (normalisé)',
    template='plotly_dark', height=450,
)
fig_radar.show()

## 4. Projections 2030 / 2050 / 2100

In [ ]:
# Génération des projections avec le meilleur modèle
best_models_map = {
    'SARIMA': arima,
    'Prophet': prophet,
    'LSTM': lstm,
    'XGBoost': xgb_model,
}

all_projections = []
for model_name, model_obj in best_models_map.items():
    proj = model_obj.generate_projections(df)
    proj['model'] = model_name
    all_projections.append(proj)

df_proj = pd.concat(all_projections, ignore_index=True)
print(f'✅ {len(df_proj)} projections générées')
df_proj.pivot_table(index=['model', 'annee'], columns='scenario', values='prediction').round(2)

In [ ]:
# Graphique projections avec intervalles de confiance
fig_proj = go.Figure()

# Historique
df_hist = df[df['annee'] >= 1950].dropna(subset=[TARGET])
fig_proj.add_trace(go.Scatter(
    x=df_hist['annee'], y=df_hist[TARGET],
    mode='lines', name='Historique (1950-2024)',
    line=dict(color='#636e72', width=2),
))

# Projections consensus (moyenne des modèles)
for scenario_name, sc_info in SCENARIOS.items():
    sub = df_proj[df_proj['scenario'] == scenario_name].groupby('annee').agg(
        pred_mean=('prediction', 'mean'),
        pred_std=('prediction', 'std'),
        lower_ci=('lower_ci', 'mean'),
        upper_ci=('upper_ci', 'mean'),
    ).reset_index()
    
    # Bande d'incertitude
    fig_proj.add_trace(go.Scatter(
        x=pd.concat([sub['annee'], sub['annee'].iloc[::-1]]),
        y=pd.concat([sub['upper_ci'], sub['lower_ci'].iloc[::-1]]),
        fill='toself', fillcolor=sc_info['color'] + '40',
        line=dict(color='rgba(0,0,0,0)'), showlegend=False,
    ))
    
    # Ligne centrale
    fig_proj.add_trace(go.Scatter(
        x=sub['annee'], y=sub['pred_mean'],
        mode='lines+markers',
        name=f'{scenario_name.title()} ({sc_info["label"]})',
        line=dict(color=sc_info['color'], width=3),
        marker=dict(size=10, symbol='diamond'),
    ))

fig_proj.add_vline(x=2024, line_dash='dot', line_color='white', annotation_text='Aujourd\'hui')
fig_proj.update_layout(
    title='📈 Projections climatiques France 2030 / 2050 / 2100 (consensus 4 modèles)',
    template='plotly_dark', height=550,
    xaxis_title='Année', yaxis_title='Température moyenne (°C)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)
fig_proj.show()

# Sauvegarde
df_proj.to_csv(REPORTS_DIR / 'projections_all_models.csv', index=False)
print(f'✅ Projections sauvegardées : {REPORTS_DIR / "projections_all_models.csv"}')

## 5. Feature Importance (XGBoost)

In [ ]:
# Importance des variables XGBoost
fi = xgb_model.get_feature_importance()
if not fi.empty:
    top_fi = fi.head(15)
    fig_fi = px.bar(
        top_fi, x='importance', y='feature', orientation='h',
        title='🔍 Feature Importance — XGBoost (Top 15)',
        template='plotly_dark', color='importance',
        color_continuous_scale='Viridis', height=500,
    )
    fig_fi.show()
else:
    print('Feature importance non disponible (fallback sklearn utilisé)')

## 6. Résumé des projections par scénario

In [ ]:
# Tableau de synthèse
summary = df_proj.groupby(['annee', 'scenario']).agg(
    pred_moy=('prediction', 'mean'),
    pred_min=('lower_ci', 'min'),
    pred_max=('upper_ci', 'max'),
).reset_index().round(2)

print('📊 Synthèse des projections (consensus 4 modèles)')
print('=' * 65)
for _, row in summary.iterrows():
    sc = SCENARIOS[row['scenario']]
    print(f'  {row["annee"]:.0f} | {row["scenario"]:15} ({sc["label"]}) : '
          f'{row["pred_moy"]:.2f}°C '
          f'[{row["pred_min"]:.2f} — {row["pred_max"]:.2f}°C]')

## ✅ Conclusions Modélisation

**Résultats clés :**
- Les 4 modèles convergent sur une hausse significative des températures
- Le scénario pessimiste (SSP5-8.5) projette +4.4°C d'ici 2100
- L'incertitude croît fortement au-delà de 2050

**Recommandation :** Utiliser le modèle avec le meilleur score RMSE pour le dashboard.

**Prochaine étape :** Lancer le dashboard interactif avec `make dashboard`
